# Ch 21 — 작은 챗봇으로 마감

본 책 10M 모델 + Q4_K_M GGUF 로 CLI 동화 co-writer 를 50줄 안에 완성합니다.

In [ ]:
# 필요 시 설치
# !pip install llama-cpp-python psutil

import os
import time

# GGUF 모델 경로
MODEL_PATH = "dist/tiny-tale-q4km.gguf"
if os.path.exists(MODEL_PATH):
    print(f"모델 발견: {MODEL_PATH}")
else:
    print(f"모델 없음: {MODEL_PATH}")
    print("Ch 20 에서 변환 후 진행하거나, SmolLM2-135M-Instruct GGUF 를 사용:")
    print("  !huggingface-cli download HuggingFaceTB/smollm2-135M-instruct-GGUF")
    print("                    smollm2-135m-instruct-q4_k_m.gguf --local-dir ./dist")

## 1. 컨셉 — 끝까지 굴리는 마지막 한 단계

본 책 10M 동화 모델은 **continuation 모델** — 사용자 prompt 를 받으면 그 다음을 이어 짓는 co-writer.

| 산출물 | 무엇을 입증 |
|---|---|
| 손실 곡선 | 학습이 진행됨 |
| PPL · benchmark | 능력이 측정됨 |
| **CLI 챗봇** | **모델이 진짜 동작함** |

## 4. 최소 예제 — 30줄 동화 co-writer

In [ ]:
# story_chat.py
# pip install llama-cpp-python

def run_story_chat(model_path, n_ctx=512, n_gpu_layers=0,
                   max_tokens=120, temperature=0.8, top_p=0.9, repeat_penalty=1.1):
    """동화 co-writer CLI 루프."""
    from llama_cpp import Llama

    llm = Llama(
        model_path=model_path,
        n_ctx=n_ctx,
        n_gpu_layers=n_gpu_layers,  # Apple Silicon: -1 (전체 Metal 가속)
        verbose=False,
    )

    print("=" * 60)
    print("Tiny Tale -- co-writer (빈 줄 입력 시 종료)")
    print("=" * 60)

    context = ""                                                        # 누적 컨텍스트
    while True:
        user = input("\n>>> ").strip()
        if not user:
            print("종료.")
            break

        # 사용자 입력을 누적 (대화 형식이 아닌 continuation)
        prompt = (context + " " + user).strip()
        out = llm(
            prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            repeat_penalty=repeat_penalty,
            stop=["\n\n", "<|endoftext|>"],                             # 두 줄 빈 줄 또는 EOS
        )
        text = out["choices"][0]["text"]
        print(text, end="", flush=True)

        context = (prompt + text)[-400:]                                # 최대 400자 유지

# 실행 (모델이 있을 때)
# run_story_chat(MODEL_PATH, n_gpu_layers=-1)  # Apple Silicon

print("run_story_chat 함수 정의 완료")
print()
print("실행 예시:")
print("  >>> Once upon a time, there was a little girl named")
print("   Lily. She had a small puppy called Max...")
print()
print("  >>> One day, they went to")
print("   the park together. The sun was shining brightly...")
print()
print("  >>> But suddenly,")
print("   a strong wind came. Lily's hat flew away. Max ran...")

## 5. 실전 — Sampling 파라미터의 역할

In [ ]:
# 도메인별 권장 sampling 파라미터
sampling_guide = [
    ("동화 (창의)", 0.9, 0.95, 1.1),
    ("챗봇 (균형)", 0.7, 0.9, 1.05),
    ("코드 (정확)", 0.2, 0.95, 1.0),
    ("번역", 0.0, None, None),
]

print(f"{'도메인':15s}  {'temp':6s}  {'top_p':7s}  {'repeat':7s}")
print("-" * 45)
for domain, temp, top_p, repeat in sampling_guide:
    tp = str(top_p) if top_p else '-'
    rp = str(repeat) if repeat else '-'
    print(f"{domain:15s}  {temp:<6}  {tp:<7}  {rp:<7}")

In [ ]:
# sampling_compare.py -- temperature 별 비교

# 실제 모델이 있을 때:
# prompt = "Lily found a magic"
# for temp in [0.1, 0.7, 1.5]:
#     for _ in range(3):
#         out = llm(prompt, max_tokens=30, temperature=temp, top_p=0.9)
#         print(f"  T={temp}: {prompt}{out['choices'][0]['text'][:60]}")
#     print()

print("예상 출력:")
print()
print("T=0.1: Lily found a magic flower in the garden. She was very happy...")
print("T=0.1: Lily found a magic flower in the garden. She was very happy...  <-- 거의 같음")
print("T=0.1: Lily found a magic flower in the garden. She was very happy...")
print()
print("T=0.7: Lily found a magic stone by the river. It glowed in the sun...")
print("T=0.7: Lily found a magic toy under her bed. She picked it up gently...")
print("T=0.7: Lily found a magic feather. She blew on it and it flew away...")
print()
print("T=1.5: Lily found a magic boots? jumping cloud what fun bird sky...  <-- 깨짐")
print("T=1.5: Lily found a magic skip slide tree happy purple monkey...")
print()
print("-> 0.7~0.9 가 sweet spot. 0.1 은 반복, 1.5 는 깨짐")

In [ ]:
# few_shot.py -- few-shot 으로 형식 유도
SYSTEM = """Continue the children's story in simple English.

Example 1:
Once upon a time, a rabbit found a carrot.
The carrot was huge and orange. The rabbit smiled and took it home.

Example 2:
There was a small bird who could not fly.
Every day she practiced. One day, she finally took off into the sky.

Now continue:
"""

def few_shot_generate(llm, user_prompt, system=SYSTEM, max_tokens=100, temperature=0.8):
    """few-shot system prompt + 사용자 입력."""
    prompt = system + user_prompt
    out = llm(prompt, max_tokens=max_tokens, temperature=temperature)
    return out["choices"][0]["text"]

# 실행 (모델이 있을 때)
# print(few_shot_generate(llm, "Lily saw a rainbow"))

print("few_shot_generate 함수 정의 완료")
print()
print("효과: 패턴 인식으로 동화 형식 유도")
print("  본 책 동화 모델은 어차피 동화 분포라 큰 차이 없지만")
print("  다른 도메인 (레시피, 코드) 학습 모델에선 효과 큼")
print()
print("캡스톤 / Part 7 LoRA 모델 -- instruction-tuned 라 system prompt + chat template 가 표준")

In [ ]:
# 성능 측정 유틸리티
import time

def measure_throughput(llm, prompt, max_tokens=100, n_runs=3):
    """처리량 (토큰/초) 측정."""
    times = []
    for _ in range(n_runs):
        start = time.time()
        out = llm(prompt, max_tokens=max_tokens, temperature=0.8)
        elapsed = time.time() - start
        times.append(max_tokens / elapsed)  # 근사치
    return sum(times) / len(times)

def measure_rss_mb():
    """현재 프로세스 RSS 메모리 (MB)."""
    import psutil
    proc = psutil.Process(os.getpid())
    return proc.memory_info().rss / 1024 / 1024

# rss = measure_rss_mb()
# print(f"현재 RSS: {rss:.1f} MB")

print("성능 측정 함수 정의 완료")
print()
print("처리량 기준:")
print("  M2 MacBook, 10M Q4_K_M: ~400 토큰/초")
print("  M2 MacBook, 1B Q4_K_M:  ~50 토큰/초")
print()
print("Apple Silicon 가속:")
print("  Llama(model_path=..., n_gpu_layers=-1)  -> 100배 빠름")

## 연습문제

1. 본인 모델로 `run_story_chat` 를 돌려 5번 대화. 모델이 어디서 깨지는가?
2. **temp 0.5 / 0.8 / 1.2** 로 같은 prompt 5개 생성. 평균 길이 + 다양성 (Jaccard) 비교.
3. `repeat_penalty` 를 1.0 / 1.1 / 1.3 로 변경. 같은 단어 반복 횟수가 어떻게 변하나?
4. `llama-cpp-python` 의 `n_gpu_layers=-1` 적용 전·후 처리량 비교 (Apple Silicon 또는 CUDA).
5. **(생각해볼 것)** 본 책 모델 (continuation) 과 SmolLM2-360M-Instruct (instruction) 의 챗봇 사용성 차이는? 어느 쪽이 사람에게 더 자연스러운가?

In [ ]:
# 연습 1: 5번 대화 + 깨지는 지점 관찰


In [ ]:
# 연습 2: temperature 별 길이 + Jaccard 다양성 비교
def jaccard_similarity(a, b):
    """두 텍스트의 단어 집합 Jaccard 유사도."""
    set_a = set(a.lower().split())
    set_b = set(b.lower().split())
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)

def avg_pairwise_jaccard(texts):
    pairs = [(i, j) for i in range(len(texts)) for j in range(i+1, len(texts))]
    sims = [jaccard_similarity(texts[i], texts[j]) for i, j in pairs]
    return sum(sims) / len(sims) if sims else 0.0

# 예시 (실제 생성 결과로 교체)
texts_low = [
    "Lily found a magic flower in the garden.",
    "Lily found a magic flower in the garden.",
    "Lily found a magic flower in the garden.",
]
texts_high = [
    "Lily found a magic stone by the river.",
    "Lily found a magic toy under her bed.",
    "Lily found a magic feather near the tree.",
]

print(f"temp=0.1 평균 Jaccard: {avg_pairwise_jaccard(texts_low):.3f}  (1.0 = 동일, 낮을수록 다양)")
print(f"temp=0.8 평균 Jaccard: {avg_pairwise_jaccard(texts_high):.3f}")

In [ ]:
# 연습 3: repeat_penalty 변화 관찰


In [ ]:
# 연습 4: n_gpu_layers 전후 처리량 비교


In [ ]:
# 연습 5: continuation vs instruction 모델 비교
